# Text Summarization: Seq2Seq + Attention (PyTorch)

Same dataset and hyperparameters as the from-scratch NumPy notebook
(`text_summ_rough.ipynb`), but the encoder/decoder LSTM cells, attention,
autograd and Adam optimizer are all provided by PyTorch — no manual
forward/backward math.

## 1. Setup & hyperparameters

In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42)
torch.manual_seed(42)

NUM_SAMPLES = 10
MAX_ARTICLE_LEN = 20
MAX_SUMMARY_LEN = 5
EMB_DIM = 16
HID_DIM = 32
EPOCHS = 100
LEARNING_RATE = 0.01
GRAD_CLIP = 5.0
TRAIN_FRACTION = 0.8


## 2. Load a small subset of the BBC News Summary dataset

In [2]:
DATA_DIR = os.path.join("..", "BBC News Summary")
ARTICLES_DIR = os.path.join(DATA_DIR, "News Articles")
SUMMARIES_DIR = os.path.join(DATA_DIR, "Summaries")

def load_all_pairs():
    pairs = []
    for category in sorted(os.listdir(ARTICLES_DIR)):
        art_dir = os.path.join(ARTICLES_DIR, category)
        sum_dir = os.path.join(SUMMARIES_DIR, category)
        if not os.path.isdir(art_dir):
            continue
        for fname in sorted(os.listdir(art_dir)):
            art_path = os.path.join(art_dir, fname)
            sum_path = os.path.join(sum_dir, fname)
            if not os.path.exists(sum_path):
                continue
            with open(art_path, encoding="latin-1") as f:
                lines = f.read().strip().split("\n")
            article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
            with open(sum_path, encoding="latin-1") as f:
                summary = f.read().strip()

            article_words = article.lower().split()[:MAX_ARTICLE_LEN]
            summary_words = summary.lower().split()[:MAX_SUMMARY_LEN]
            if len(article_words) < 5 or len(summary_words) < 3:
                continue
            pairs.append((" ".join(article_words), " ".join(summary_words)))
    return pairs

all_pairs = load_all_pairs()
random.shuffle(all_pairs)
pairs = all_pairs[:NUM_SAMPLES]

split = int(TRAIN_FRACTION * len(pairs))
train_pairs = pairs[:split]
test_pairs = pairs[split:]

print(f"Total usable pairs in dataset: {len(all_pairs)}")
print(f"Using {len(pairs)} pairs -> {len(train_pairs)} train / {len(test_pairs)} test")
print("\nExample pair:")
print("Article: ", train_pairs[0][0])
print("Summary: ", train_pairs[0][1])


Total usable pairs in dataset: 2225
Using 10 pairs -> 8 train / 2 test

Example pair:
Article:  the uk property market remains robust despite the recent slowdown, according to mortgage lender bradford & bingley and housebuilder george
Summary:  wimpey said the uk housing


## 3. Build the vocabulary and encode text as integer ids

In [3]:
PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
SPECIAL_TOKENS = [PAD, SOS, EOS, UNK]

vocab_words = set()
for article, summary in pairs:
    vocab_words.update(article.split())
    vocab_words.update(summary.split())

itos = SPECIAL_TOKENS + sorted(vocab_words)
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)
print(f"Vocabulary size: {VOCAB_SIZE}")

def encode(text, add_sos=False, add_eos=False):
    ids = [stoi[SOS]] if add_sos else []
    ids += [stoi.get(w, stoi[UNK]) for w in text.split()]
    if add_eos:
        ids.append(stoi[EOS])
    return torch.tensor(ids, dtype=torch.long)

train_data = [(encode(a), encode(b, add_sos=True, add_eos=True)) for a, b in train_pairs]
test_data = [(encode(a), encode(b, add_sos=True, add_eos=True)) for a, b in test_pairs]


Vocabulary size: 177


## 4. Model: LSTM encoder + Bahdanau attention + LSTM decoder

A single `nn.Module` holding everything. `nn.LSTM` runs the whole encoder in one
call and returns every hidden state (needed by attention) plus the final `(h, c)`,
which is bridged into an `nn.LSTMCell` decoder run step-by-step so attention can be
recomputed each step — same architecture as the NumPy version, just with no
hand-written gate math or backward pass.

In [4]:
class Seq2SeqAttn(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.emb_enc = nn.Embedding(vocab_size, emb_dim)
        self.emb_dec = nn.Embedding(vocab_size, emb_dim)
        self.encoder = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.decoder_cell = nn.LSTMCell(emb_dim + hid_dim, hid_dim)

        # Bahdanau (additive) attention: score = va . tanh(Wa @ s_prev + Ua @ h_i)
        self.Wa = nn.Linear(hid_dim, hid_dim, bias=False)
        self.Ua = nn.Linear(hid_dim, hid_dim, bias=False)
        self.va = nn.Linear(hid_dim, 1, bias=False)

        self.out = nn.Linear(hid_dim, vocab_size)

    def encode(self, x_ids):
        emb = self.emb_enc(x_ids).unsqueeze(0)          # [1, T_x, emb_dim]
        H, (h, c) = self.encoder(emb)
        return H.squeeze(0), h[0, 0], c[0, 0]            # H: [T_x, hid], h/c: [hid]

    def attend(self, s_prev, H):
        scores = self.va(torch.tanh(self.Wa(s_prev) + self.Ua(H))).squeeze(-1)  # [T_x]
        alpha = F.softmax(scores, dim=0)
        context = alpha @ H                                                     # [hid_dim]
        return context

    def decode_step(self, y_prev_id, s_prev, c_prev, H):
        context = self.attend(s_prev, H)
        x_t = torch.cat([self.emb_dec(y_prev_id), context]).unsqueeze(0)
        s_t, c_t = self.decoder_cell(x_t, (s_prev.unsqueeze(0), c_prev.unsqueeze(0)))
        s_t, c_t = s_t[0], c_t[0]
        logits = self.out(s_t)
        return logits, s_t, c_t

    def forward(self, x_ids, y_ids):
        H, s_prev, c_prev = self.encode(x_ids)           # bridge: encoder's last (h, c)
        loss = 0.0
        for t in range(len(y_ids) - 1):
            logits, s_prev, c_prev = self.decode_step(y_ids[t], s_prev, c_prev, H)
            loss = loss + F.cross_entropy(logits.unsqueeze(0), y_ids[t + 1].unsqueeze(0))
        return loss / max(1, len(y_ids) - 1)

    @torch.no_grad()
    def generate(self, x_ids, sos_id, eos_id, max_len):
        H, s_prev, c_prev = self.encode(x_ids)
        y_id = torch.tensor(sos_id)
        words = []
        for _ in range(max_len):
            logits, s_prev, c_prev = self.decode_step(y_id, s_prev, c_prev, H)
            y_id = logits.argmax()
            if y_id.item() == eos_id:
                break
            words.append(y_id.item())
        return words


## 5. Training loop

In [5]:
model = Seq2SeqAttn(VOCAB_SIZE, EMB_DIM, HID_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train_data)
    total_loss = 0.0
    for x_ids, y_ids in train_data:
        optimizer.zero_grad()
        loss = model(x_ids, y_ids)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
    if epoch == 1 or epoch % 20 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  avg loss: {total_loss / len(train_data):.4f}")


Epoch   1/100  avg loss: 5.1196
Epoch  20/100  avg loss: 0.1322
Epoch  40/100  avg loss: 0.0292
Epoch  60/100  avg loss: 0.0139
Epoch  80/100  avg loss: 0.0084
Epoch 100/100  avg loss: 0.0057


## 6. Greedy inference

In [6]:
def summarize(article_text, max_len=MAX_SUMMARY_LEN + 2):
    x_ids = encode(article_text)
    word_ids = model.generate(x_ids, stoi[SOS], stoi[EOS], max_len)
    return " ".join(itos[i] for i in word_ids)


## 7. Evaluate on held-out test articles

In [7]:
print("=" * 60)
print("TEST ON HELD-OUT BBC NEWS ARTICLES")
print("=" * 60)
for article, reference in test_pairs:
    generated = summarize(article)
    print(f"\nArticle:   {article}")
    print(f"Reference: {reference}")
    print(f"Generated: {generated}")


TEST ON HELD-OUT BBC NEWS ARTICLES

Article:   women's football legend mia hamm has played her final game. hamm, 32, who officially retired after this year's athens olympics,
Reference: women's football legend mia hamm
Generated: wimpey said the uk housing

Article:   microsoft has said it will replace more than 14 million power cables for its xbox consoles due to safety concerns.
Reference: microsoft has said it will
Generated: but now he has pulled
